# GPUs and why they matter

Lena trains a model on her laptop to sort 64 by 64 grey product photos into 10 categories, and it runs all night. In this notebook you count the work in one training step, time the same step on the processor (CPU) and on a GPU, trace the training loop on the GPU, and fix the real device error.

About 5 minutes. **First, in the menu: Runtime > Change runtime type > pick a GPU (for example T4).** The notebook also runs on a plain CPU runtime, but then the GPU cells say they are skipped. No account or key needed.

The saved outputs below come from our run on a Mac, where PyTorch reaches the GPU through the device name `mps`. On Colab's GPU you will see `cuda` instead, and your own times.

## 1. Which device do we have?

The next cell picks the GPU when PyTorch can see one, and the CPU otherwise. On Colab and NVIDIA cards the GPU is `"cuda"`; on a Mac it is `"mps"`.

Look for: the line `GPU device:`. If it says `none`, the runtime has no GPU: go back to the menu step above.

In [1]:
import json, statistics, sys, time
import torch
from torch import nn

torch.manual_seed(0)

if torch.cuda.is_available():
    GPU = "cuda"
elif torch.backends.mps.is_available():
    GPU = "mps"   # a Mac's GPU
else:
    GPU = None    # no GPU in this runtime

print("torch version:", torch.__version__)
print("GPU device:", GPU or "none (CPU only)")
if GPU == "cuda":
    print("GPU name:", torch.cuda.get_device_name(0))

torch version: 2.15.0
GPU device: mps


## 2. Count the work in one step

Lena's network has three plain layers: 4096 -> 2048 -> 2048 -> 10. A layer with n inputs and m outputs does n x m multiply-adds per photo. One step feeds a batch of 256 photos forward, and the backward pass costs about twice the forward pass.

Look for: about **9.7 billion** multiply-adds per step, and 100 epochs over 200,000 photos is **78,200** steps.

In [2]:
NET = [4096, 2048, 2048, 10]
BATCH = 256
PHOTOS, EPOCHS = 200_000, 100   # Lena's job (story numbers)

layer_work = [NET[i] * NET[i + 1] for i in range(len(NET) - 1)]
per_photo = sum(layer_work)
forward = per_photo * BATCH
step_work = forward * 3          # forward + backward (about twice the forward)
steps_job = -(-PHOTOS // BATCH) * EPOCHS

for i, w in enumerate(layer_work):
    print(f"layer {i + 1} ({NET[i]} x {NET[i + 1]}): {w:,} multiply-adds per photo")
print(f"one photo, all layers:   {per_photo:,}")
print(f"forward, batch {BATCH}:     {forward / 1e9:.1f} billion")
print(f"one training step:       {step_work / 1e9:.1f} billion")
print(f"steps in Lena's job:     {steps_job:,}")

layer 1 (4096 x 2048): 8,388,608 multiply-adds per photo
layer 2 (2048 x 2048): 4,194,304 multiply-adds per photo
layer 3 (2048 x 10): 20,480 multiply-adds per photo
one photo, all layers:   12,603,392
forward, batch 256:     3.2 billion
one training step:       9.7 billion
steps in Lena's job:     78,200


## 3. A fair timer

This is the timing code from the lesson's run. It builds the network on a device, moves the batch to the device inside every step (like a real loop), runs 5 warm-up steps, and returns the median time of 5 repeats.

One detail matters: PyTorch hands work to the GPU and moves on before the GPU is done. So `sync` waits for the GPU to finish before the clock stops. Without it the clock measures too little.

Look for: the network has **12,607,498** weights and biases, about **50 MB** as 4-byte numbers.

In [3]:
def make_net(sizes):
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i + 1]))
        if i < len(sizes) - 2:
            layers.append(nn.ReLU())
    return nn.Sequential(*layers)


def sync(device):
    if device == "cuda":
        torch.cuda.synchronize()
    elif device == "mps":
        torch.mps.synchronize()


def time_steps(sizes, batch, device, steps, repeats=5, wait=True):
    """Median milliseconds per training step (batch moved to the device inside the step)."""
    torch.manual_seed(0)
    model = make_net(sizes).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()
    xb_cpu = torch.randn(batch, sizes[0])
    yb_cpu = torch.randint(0, sizes[-1], (batch,))

    def step():
        xb, yb = xb_cpu.to(device), yb_cpu.to(device)
        loss = loss_fn(model(xb), yb)
        opt.zero_grad()
        loss.backward()
        opt.step()

    for _ in range(5):  # warm-up: the first GPU calls compile and allocate
        step()
    sync(device)
    per_step = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        for _ in range(steps):
            step()
        if wait:
            sync(device)
        per_step.append((time.perf_counter() - t0) / steps)
    return statistics.median(per_step) * 1000


params = sum(p.numel() for p in make_net(NET).parameters())
print(f"weights and biases: {params:,} (about {params * 4 / 1e6:.0f} MB as float32)")

weights and biases: 12,607,498 (about 50 MB as float32)


## 4. The step on the CPU

The next cell times Lena's step on the processor. It takes about half a minute.

Look for: the milliseconds per step, the speed in billions of multiply-adds per second, and how many hours her whole job would take.

The lesson showed 296 ms per step and about 6.4 hours. We measured that while our Mac was busy with other jobs, which slowed its processor much more than its GPU. The saved output below is the same Mac when it was quieter, and much faster. Your numbers depend on the machine Colab gives you.

In [4]:
cpu_ms = time_steps(NET, BATCH, "cpu", steps=10)
print(f"CPU: {cpu_ms:.1f} ms per step")
print(f"CPU speed: {step_work / (cpu_ms / 1000) / 1e9:.1f} billion multiply-adds per second")
print(f"Lena's job on the CPU: {steps_job * cpu_ms / 1000 / 3600:.1f} hours")

CPU: 50.8 ms per step
CPU speed: 190.5 billion multiply-adds per second
Lena's job on the CPU: 1.1 hours


## 5. The same step on the GPU

Same network, same batch, same code, on the GPU.

Look for: a shorter time per step, and the line `GPU is ... times faster` with a number above 1.

The lesson showed about 23 ms, about 13 times faster, from the busy Mac. In the saved output below, from the quieter Mac, the GPU is only about 2.5 times faster: a Mac's processor is strong at this kind of math once nothing else slows it down. Colab gives its own numbers.

In [5]:
if GPU is None:
    gpu_ms = None
    print("No GPU in this runtime: skipped. Switch the runtime to a GPU and run again.")
else:
    gpu_ms = time_steps(NET, BATCH, GPU, steps=10)
    print(f"GPU ({GPU}): {gpu_ms:.1f} ms per step")
    print(f"GPU speed: {step_work / (gpu_ms / 1000) / 1e9:.1f} billion multiply-adds per second")
    print(f"GPU is {cpu_ms / gpu_ms:.1f} times faster than the CPU here")
    print(f"Lena's job on the GPU: {steps_job * gpu_ms / 1000 / 60:.0f} minutes")

GPU (mps): 20.4 ms per step
GPU speed: 474.0 billion multiply-adds per second
GPU is 2.5 times faster than the CPU here
Lena's job on the GPU: 27 minutes


## 6. Does the GPU always win?

The win comes from running many multiply-adds at once, so it should depend on how much work each step has. The next cell times two smaller cases on both devices: the same network at batch 16, and a tiny 20-16-2 network at batch 32.

Look for: at batch 16 the GPU is still ahead, but by less. For the tiny network the line ends in `times SLOWER` (the lesson showed about 3.8 times; the saved run below shows more). Every step pays a fixed cost to copy the batch and start the work on the GPU, and with so little work that cost is most of the step.

In [6]:
TINY = [20, 16, 2]
cases = [("4096-2048-2048-10, batch 16", NET, 16, 20), ("20-16-2, batch 32", TINY, 32, 200)]
for name, sizes, batch, steps in cases:
    work = sum(sizes[i] * sizes[i + 1] for i in range(len(sizes) - 1)) * batch * 3
    c = time_steps(sizes, batch, "cpu", steps)
    if GPU is None:
        print(f"{name}: {work:,} multiply-adds per step | CPU {c:.1f} ms | GPU skipped")
        continue
    g = time_steps(sizes, batch, GPU, steps)
    verdict = f"GPU {c / g:.1f} times faster" if g < c else f"GPU {g / c:.1f} times SLOWER"
    print(f"{name}: {work:,} multiply-adds per step | CPU {c:.1f} ms | GPU {g:.1f} ms | {verdict}")

4096-2048-2048-10, batch 16: 604,962,816 multiply-adds per step | CPU 21.6 ms | GPU 11.1 ms | GPU 1.9 times faster


20-16-2, batch 32: 33,792 multiply-adds per step | CPU 0.4 ms | GPU 2.9 ms | GPU 6.8 times SLOWER


## 7. The training loop on the GPU, traced

This is the training loop you know, with two changes: the model moves to the device once, and every batch moves inside the loop. The next cell runs it on two batches of 256 random "photos" and prints, for each line about to run, where the model, the photos and the labels live.

Look for: the model starts `on cpu` and is on the GPU (`cuda:0` on Colab, `mps:0` on our Mac) after `model.to(device)`. Each new batch arrives on the CPU and moves line by line. At the end, `next(model.parameters()).device` names the GPU.

In [7]:
torch.manual_seed(0)
device = GPU or "cpu"
model = nn.Sequential(nn.Linear(4096, 2048), nn.ReLU(), nn.Linear(2048, 2048), nn.ReLU(), nn.Linear(2048, 10))
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.SGD(model.parameters(), lr=0.01)
batches = [(torch.randn(256, 4096), torch.randint(0, 10, (256,))) for _ in range(2)]


def train_one_epoch(model, batches, device):
    model.to(device)
    for photos, labels in batches:
        photos = photos.to(device)
        labels = labels.to(device)
        loss = loss_fn(model(photos), labels)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return loss.item()


def where(v):
    if isinstance(v, nn.Module):
        return str(next(v.parameters()).device)
    if isinstance(v, torch.Tensor):
        return str(v.device) if v.dim() else f"{v.item():.4f} on {v.device}"
    return "-"


def trace_run(fn, *args):
    """Run fn(*args) and print each line just before it runs, with where model, photos, labels, loss live."""
    import inspect
    src = inspect.getsource(fn).splitlines()
    first = fn.__code__.co_firstlineno
    def local(frame, event, arg):
        if event in ("line", "return"):
            n = frame.f_lineno - first
            v = frame.f_locals
            text = "(return)" if event == "return" else src[n].strip()
            print(f"{text:<40} model {where(v.get('model')):<6} photos {where(v.get('photos')):<6} "
                  f"labels {where(v.get('labels')):<6} loss {where(v.get('loss'))}")
        return local
    sys.settrace(lambda frame, event, arg: local if frame.f_code is fn.__code__ else None)
    try:
        return fn(*args)
    finally:
        sys.settrace(None)


result = trace_run(train_one_epoch, model, batches, device)
print("returned:", result)
print("next(model.parameters()).device:", next(model.parameters()).device)

model.to(device)                         model cpu    photos -      labels -      loss -
for photos, labels in batches:           model mps:0  photos -      labels -      loss -
photos = photos.to(device)               model mps:0  photos cpu    labels cpu    loss -
labels = labels.to(device)               model mps:0  photos mps:0  labels cpu    loss -
loss = loss_fn(model(photos), labels)    model mps:0  photos mps:0  labels mps:0  loss -
opt.zero_grad()                          model mps:0  photos mps:0  labels mps:0  loss 2.2980 on mps:0
loss.backward()                          model mps:0  photos mps:0  labels mps:0  loss 2.2980 on mps:0
opt.step()                               model mps:0  photos mps:0  labels mps:0  loss 2.2980 on mps:0
for photos, labels in batches:           model mps:0  photos mps:0  labels mps:0  loss 2.2980 on mps:0
photos = photos.to(device)               model mps:0  photos cpu    labels cpu    loss 2.2980 on mps:0
labels = labels.to(device)              

## 8. Broken on purpose: the batch stays on the CPU

A first try at the GPU version moves the model and nothing else. The cell catches the error and prints its message, so the notebook keeps running.

Look for: a `RuntimeError` that names two devices, the CPU and the GPU. On our Mac it reads "Tensor for argument input is on cpu but expected on mps". On Colab's GPU the wording is different (it says it found tensors on `cuda:0` and `cpu`); look for the two device names.

In [8]:
def train_one_epoch_broken(model, batches, device):
    model.to(device)
    for photos, labels in batches:
        loss = loss_fn(model(photos), labels)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return loss.item()


try:
    train_one_epoch_broken(model, batches, device)
    print("No error: model and batches are all on", device, "(no GPU in this runtime, so nothing is on two devices)")
except RuntimeError as e:
    print("RuntimeError:", e)

RuntimeError: Tensor for argument input is on cpu but expected on mps


## 9. Half a fix: only the photos move

Now the photos move, but the labels stay on the CPU.

Look for: the photos error is gone, and a new `RuntimeError` appears in the loss. On our Mac: "target must be an MPS tensor". The labels are part of the batch too.

In [9]:
def train_one_epoch_labels(model, batches, device):
    model.to(device)
    for photos, labels in batches:
        photos = photos.to(device)
        loss = loss_fn(model(photos), labels)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return loss.item()


try:
    train_one_epoch_labels(model, batches, device)
    print("No error: everything is on", device, "(no GPU in this runtime)")
except RuntimeError as e:
    print("RuntimeError:", e)

RuntimeError: target must be an MPS tensor


## 10. The fix

Move the photos **and** the labels as the first line inside the loop, as the traced loop did. When you see two device names in an error, print `.device` of each tensor on the failing line.

Look for: no error, a printed loss, and the device of the model and of the last batch: the same GPU device.

In [10]:
def train_one_epoch_fixed(model, batches, device):
    model.to(device)
    for photos, labels in batches:
        photos, labels = photos.to(device), labels.to(device)
        loss = loss_fn(model(photos), labels)
        opt.zero_grad()
        loss.backward()
        opt.step()
    print("model on:", next(model.parameters()).device, "| last photos on:", photos.device, "| last labels on:", labels.device)
    return loss.item()


print("loss:", round(train_one_epoch_fixed(model, batches, device), 4))

model on: mps:0 | last photos on: mps:0 | last labels on: mps:0


loss: 2.2931


## Now change it

Each cell below has one line marked `# <- edit`. Change it, run the cell, and compare.

**Experiment 1: a smaller batch.** Change `batch` to 64, then to 16. Each step has less work, so there is less to run side by side: the GPU stays ahead but its lead (the `times faster` number) shrinks.

In [11]:
batch = 256  # <- edit: try 64, then 16
c = time_steps(NET, batch, "cpu", steps=10)
if GPU is None:
    print(f"batch {batch}: CPU {c:.1f} ms | GPU skipped (no GPU)")
else:
    g = time_steps(NET, batch, GPU, steps=10)
    print(f"batch {batch}: CPU {c:.1f} ms | GPU {g:.1f} ms | GPU {c / g:.1f} times faster")

batch 256: CPU 66.9 ms | GPU 19.3 ms | GPU 3.5 times faster


**Experiment 2: give the tiny network more work.** Change `sizes` to `[20, 2048, 2048, 2]`. Two big hidden layers add millions of multiply-adds per step, so the GPU has enough work to run side by side: the line should flip from `times SLOWER` to `times faster`.

In [12]:
sizes = [20, 16, 2]  # <- edit: try [20, 2048, 2048, 2]
work = sum(sizes[i] * sizes[i + 1] for i in range(len(sizes) - 1)) * 32 * 3
c = time_steps(sizes, 32, "cpu", steps=20)
if GPU is None:
    print(f"{sizes}, batch 32: {work:,} multiply-adds per step | CPU {c:.1f} ms | GPU skipped")
else:
    g = time_steps(sizes, 32, GPU, steps=20)
    verdict = f"GPU {c / g:.1f} times faster" if g < c else f"GPU {g / c:.1f} times SLOWER"
    print(f"{sizes}, batch 32: {work:,} multiply-adds per step | CPU {c:.1f} ms | GPU {g:.1f} ms | {verdict}")

[20, 16, 2], batch 32: 33,792 multiply-adds per step | CPU 0.7 ms | GPU 3.3 ms | GPU 4.7 times SLOWER


**Experiment 3: the old notebook with `device = "cpu"` written at the top.** Change `device_used` to `"cpu"`. The runtime still has a GPU, but the model and every batch now stay on the processor: the step is as slow as on the CPU in step 4. Having a GPU is not enough; only `.to(device)` puts work on it.

In [13]:
device_used = GPU or "cpu"  # <- edit: try "cpu"
m = make_net(NET).to(device_used)
print("model on:", next(m.parameters()).device)
print(f"step time with device_used={device_used!r}: {time_steps(NET, BATCH, device_used, steps=10):.1f} ms")

model on: mps:0


step time with device_used='mps': 18.4 ms


## What you should see

- The first cell prints `GPU device: cuda` on a Colab GPU runtime (ours, on a Mac: `mps`). `none` means the runtime has no GPU.
- For the 4096-2048-2048-10 network at batch 256, the GPU line prints a shorter time than the CPU line and a `times faster` number above 1 (ours: 2.5).
- For the tiny 20-16-2 network at batch 32, the line ends in `times SLOWER`: the GPU is not faster.
- In the trace, the model is on `cpu` before `model.to(device)` and on the GPU after it, and `next(model.parameters()).device` names the GPU.
- The loop that never moves the batch prints a `RuntimeError` that names the CPU and the GPU; the fixed loop prints a loss and no error.